# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado
  
El objetivo de este ejercicio es conseguir extraer información valiosa de las reseñas de un dataset más 'crudo' o que necesita un preprocesado mayor que el del ejercicio 1.

1. **Filtramos por las 100 reseñas con el contenido más largo**. Esto ya os lo doy hecho más abajo.
2. En el DataFrame resultante, habremos obtenido muchas filas en las que realmente el contenido de la reseña no es nada relevante. **Instruir vía Prompting a un LLM para que filtre las filas del DataFrame que de verdad aportan información de la que se pueden extraer insights relevantes**
3. **Paso final:** extraer información relevante de este último dataframe doblemente filtrado usando un segundo LLM. Por ejemplo, un json final como este:

    ```json
    {
    "Id": "{{id}}",
    "SentimientoGeneral": "Positivo|Negativo|Neutral",
    "AspectosPositivos": "[lista de aspectos positivos mencionados]",
    "AspectosNegativos": "[lista de aspectos negativos mencionados]",
    "Dificultad": "Demasiado Fácil|Fácil|Equilibrado|Difícil|Demasiado Difícil|No Mencionado",
    "Recomendado": "{{recomendado}}"
    }
    ```

**Consideraciones adicionales:**

- También tienes un csv de ejemplo de resultado final (analisis_videojuegos_resultados.csv) donde se han extraído otras entidades y se ha entregado otro formato. **Esto es a libre elección, pero un mínimo de 3 entidades extraídas** por el segundo LLM.
- Los modelos suelen ser buenos realizando tareas de una en una. Por eso separamos en dos pasos el filtrado de contenido relevante y el de extracción de ese contenido.
- Puedes usar cualquier proveedor y canal: vía API (como Groq/Gemini) o en local vía HuggingFce teniendo en cuenta las limitaciones y ventajas de cada uno, rate limits del modelo elegido, tamaño... 
- Tener en cuenta qué cantidad de tokens/información/tareas le estamos pasando al LLM en cada llamada comparándolo con los límites que tenemos para el LLM elegido
- Se valorará **justificar cada elección** como: cada time.stop() añadido, cada lote de filas de DataFrame que pasemos en cada llamada... etc ajustándonos como digo a esos límites que tenemos.

In [1]:
import pandas as pd
import time
import re

In [2]:
data = pd.read_csv('videogames_reviews.csv')
data

,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0
...,...,...,...,...
19995,19995,si,Recomendado,1
19996,19996,cojonudo,Recomendado,1
19997,19997,reostia historia guapisima graficos impresiona...,Recomendado,1
19998,19998,basicamente sublime obra maestra,Recomendado,1


In [3]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0             int64
Contenido                str
Valoración               str
Recomendado_binario    int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [4]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 100 comentarios con más texto para análisis posterior.

In [5]:
# Preprocesamiento simplificado: El Top 100 de los comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminamos filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calculamos la longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionamos los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Eliminamos columnas innecesarias
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609
...,...,...,...,...
95,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
96,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
97,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516
98,after hearing all the fuss about skyrim for ma...,No recomendado,0,5458


## **Paso 2: Continúa...**

## **PASO 2 — Filtrado de reseñas relevantes mediante LLM**

Tras obtener las 100 reseñas más largas, el siguiente paso consiste en filtrar cuáles de ellas contienen información realmente útil para extraer insights.

Muchas reseñas largas no aportan valor (insultos, frases sin sentido, quejas vagas, etc.).  
Por ello, utilizamos un **primer LLM** cuya única tarea es decidir si una reseña:

- aporta información sobre el juego  
- menciona aspectos como jugabilidad, rendimiento, historia, gráficos, bugs, dificultad, etc.  
- permite extraer insights accionables  

Este paso se realiza **por lotes** para evitar superar los límites de tokens del modelo y para optimizar el coste computacional.


#### Inicialización del cliente Groq para usar LLM
Usamos este cliente debido a que el de Gemini el API Key no puede recibir mas de 10 llamados por minutos en caso de tener API Key en modo free

In [6]:
pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]2m1/2 [openai]
Note: you may need to restart the kernel to use updated packages.


In [7]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("Cliente OpenRouter inicializado correctamente.")


Cliente OpenRouter inicializado correctamente.


In [8]:
prompt_filtrado = """
Eres un analista experto en reseñas de videojuegos.

Tu tarea es leer un lote de reseñas y devolver SOLO los índices de aquellas que:
- contienen información útil sobre el juego
- mencionan aspectos como jugabilidad, rendimiento, historia, gráficos, bugs, dificultad, contenido, etc.
- permiten extraer insights accionables

NO consideres relevantes:
- insultos sin contexto
- reseñas de 1 a 5 palabras
- frases sin información (“corre”, “mal optimizado”, “nose parece chotada”)
- spam o texto incoherente

Formato de salida (JSON estricto):
{
  "relevantes": [lista de índices]
}

Aquí tienes las reseñas:

{{RESEÑAS}}
"""


#### Funcion Filtramos por Lotes

In [9]:
import re

def filtro_previo(df):
    patrones_utiles = [
        r"bug", r"error", r"crash", r"rendimiento", r"fps",
        r"historia", r"gráfic", r"jugabil", r"misión", r"contenido",
        r"optimiza", r"difícil", r"balance"
    ]

    def es_util(texto):
        if len(texto) < 80:
            return False
        texto_lower = texto.lower()
        return any(re.search(p, texto_lower) for p in patrones_utiles)

    df_filtrado = df[df["Contenido"].apply(es_util)]
    return df_filtrado.reset_index(drop=True)

df_pre_filtrado = filtro_previo(df_top_100)
df_pre_filtrado

,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,this was probably my first preorder i felt tha...,Recomendado,1,7662
1,oh well admittedly its difficult for to write ...,No recomendado,0,7638
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,i had enormous expectations having put thousan...,No recomendado,0,7609
4,i finished the witcher 3 for the first time th...,Recomendado,1,7604
...,...,...,...,...
77,shovelware imitation of game that never actual...,No recomendado,0,5617
78,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
79,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
80,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516


In [12]:
def filtrar_llm(df, client):
    indices = []

    for idx, row in df.iterrows():
        reseña = row["Contenido"][:1500]

        # Usamos el MISMO prompt que definiste antes
        prompt = prompt_filtrado.replace("{{RESEÑAS}}", f"{idx}: {reseña}")

        try:
            response = client.chat.completions.create(
                model="mistralai/mixtral-8x22b-instruct",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )

            content = response.choices[0].message.content
            relevantes = eval(content)["relevantes"]

            if idx in relevantes:
                indices.append(idx)

        except Exception as e:
            print(f"Error en reseña {idx}: {e}")

        time.sleep(0.3)

    return df.loc[indices]


In [13]:
df_filtrado = filtrar_llm(df_pre_filtrado, client)

Error en reseña 0: invalid syntax (<string>, line 1)
Error en reseña 1: invalid syntax (<string>, line 1)
Error en reseña 2: invalid syntax (<string>, line 1)
Error en reseña 3: invalid syntax (<string>, line 1)
Error en reseña 4: invalid syntax (<string>, line 1)


KeyboardInterrupt: 

#### Ejecutamos el filtrado

In [ ]:
print("Filtrando reseñas relevantes mediante LLM con OpenRouter...")

df_filtrado = filtrar_llm(df_pre_filtrado, client)
df_filtrado = df_filtrado.reset_index(drop=True)

print(f"Reseñas relevantes encontradas: {len(df_filtrado)}")
df_filtrado.head()

#### Guardamos el filtrado

In [ ]:
df_filtrado.to_csv("reseñas_filtradas_llm.csv", index=False)
df_filtrado

Respaldo de la celda

In [ ]:
import json
import re

def extraer_info_llm(row, client):
    reseña = row["Contenido"][:2000]  # límite de seguridad
    val = row["Valoración"]
    idx = row.name

    # Construcción del prompt
    prompt = (
        prompt_extraccion
        .replace("{{ID}}", str(idx))
        .replace("{{VAL}}", val)
        .replace("{{TEXTO}}", reseña)
    )

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",  # o tu modelo de OpenRouter
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        content = response.choices[0].message.content.strip()

        # 1. Intentar JSON directo
        entidad_json = None
        if content.startswith("{") and content.endswith("}"):
            try:
                entidad_json = json.loads(content)
            except:
                entidad_json = None

        # 2. Fallback: extraer JSON con regex
        if entidad_json is None:
            json_match = re.search(r"\{.*\}", content, re.DOTALL)
            if json_match:
                try:
                    entidad_json = json.loads(json_match.group())
                except:
                    entidad_json = None

        # Si sigue sin JSON, descartar
        if entidad_json is None:
            print(f"JSON inválido en reseña {idx}")
            return None

        # -------------------------------
        # 🔥 MEJORA: Validación de entidades
        # -------------------------------

        # Asegurar que existan los campos esperados
        campos_lista = ["AspectosPositivos", "AspectosNegativos", "TemasMencionados"]

        for campo in campos_lista:
            if campo not in entidad_json:
                entidad_json[campo] = []

            # Si está vacío, rellenar con un valor genérico
            if not entidad_json[campo]:
                entidad_json[campo] = ["No especificado"]

        # Validar campo de dificultad
        if not entidad_json.get("Dificultad"):
            entidad_json["Dificultad"] = "No mencionado"

        # Validar sentimiento
        if not entidad_json.get("SentimientoGeneral"):
            entidad_json["SentimientoGeneral"] = "Neutral"

        # Validar recomendado
        if not entidad_json.get("Recomendado"):
            entidad_json["Recomendado"] = val

        # Asegurar que el ID esté presente
        entidad_json["Id"] = idx

        return entidad_json

    except Exception as e:
        print(f"Error procesando reseña {idx}: {e}")
        return None
